### Reading the PDF and Extracting the text

In [1]:
from pypdf import PdfReader

In [2]:
text=''
try:
    reader = PdfReader('./PDFs/bom.pdf')
    for page in reader.pages:
        text+= page.extract_text()

    print(f"extracted text {text[:1000]}")
except Exception as exc:
    # TO Catch error 
    print("Remaining")

extracted text Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
       Basics of Mathematics                                                                                                                    2 
 
Section A 
A.1 Introduction to Qualitative and 
Quantitative Analysis 
Qualitative Data Analysis 
Qualitative data an

### Chunking the PDF
:-- Using recursive chunking

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        " ",
        ".",
        ",",
        "\u200b",  # Zero-width space
        "\uff0c",  # Fullwidth comma
        "\u3001",  # Ideographic comma
        "\uff0e",  # Fullwidth full stop
        "\u3002",  # Ideographic full stop
        "",
    ],
)

all_splits = text_splitter.split_text(text)

print(f"Split PDF text into {len(all_splits)} sub-documents.")


Split PDF text into 66 sub-documents.


In [7]:
for i, chunk in enumerate(all_splits):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\n--- First half of chunk {i+1} ---")
    print(chunk[:300])
    print(f"\n--- Second half of chunk {i+1} ---")
    print(chunk[300:])
    print(f"\nLength: {len(chunk)}")

    if i == 2:
        break


--- Chunk 1 ---

--- First half of chunk 1 ---
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 


--- Second half of chunk 1 ---
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
       Basics of Mathematics                                                                                                                    2 
 
Section A 
A.1 Introduction to Qualitative and 
Qua

### Embeddings

In [13]:
from sentence_transformers import SentenceTransformer

In [15]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(all_splits)

print(f"Each embedding has {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions


array([[ 0.02926199,  0.04354955, -0.00183986, ..., -0.03152945,
        -0.01072447,  0.00464847],
       [ 0.08955773, -0.00526012, -0.09272801, ...,  0.02244636,
         0.01970464, -0.06571968],
       [ 0.04630141,  0.01660525, -0.07739811, ...,  0.02257555,
         0.00361892, -0.08824003],
       ...,
       [-0.05672327, -0.03442134,  0.04211261, ...,  0.05235911,
        -0.05188822,  0.0263025 ],
       [-0.05116495, -0.02751498,  0.03495638, ...,  0.05592861,
        -0.05213397,  0.04474709],
       [-0.0564223 , -0.00289954,  0.04824543, ...,  0.07061722,
        -0.06611109,  0.03254245]], shape=(66, 384), dtype=float32)

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
similarities = cosine_similarity(embeddings)
print("Similarity matrix:")
print(similarities)
# You'll see sentences 1, 2, and 4 are more similar to each other


Similarity matrix:
[[0.9999999  0.27995884 0.32016426 ... 0.5082781  0.4807489  0.4872148 ]
 [0.27995884 0.9999999  0.76331556 ... 0.1517172  0.17621809 0.12574951]
 [0.32016426 0.76331556 1.         ... 0.24193653 0.25580803 0.23576777]
 ...
 [0.5082781  0.1517172  0.24193653 ... 1.         0.9651926  0.90530217]
 [0.4807489  0.17621809 0.25580803 ... 0.9651926  1.         0.88975585]
 [0.4872148  0.12574951 0.23576777 ... 0.90530217 0.88975585 1.        ]]
